# LLM Safety Eval Pipeline — Colab runner

Runs a HuggingFace model against jailbreak safety benchmarks (JailbreakBench, HarmBench, StrongREJECT) and pushes transparent results (every prompt, response, and score) to your GitHub repo.

**Before running:**
1. Runtime -> Change runtime type -> **T4 GPU** (free tier is fine for small models).
2. In the left sidebar, click the key icon (Secrets) and add:
   - `GITHUB_TOKEN` — a GitHub personal access token (repo scope) so this notebook can push results.
   - `HF_TOKEN` — optional, only needed if you use HarmBench (it's a gated dataset).
   - `ANTHROPIC_API_KEY` — optional, only needed if you want the higher-fidelity `llm_judge` scorer instead of the free `rule_based` one.

In [ ]:
# 1. Configure your run here
GITHUB_REPO_URL = "https://github.com/<your-username>/<your-repo>.git"  # <-- edit me
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"   # small + popular + Colab-T4-friendly. Swap for any HF model id.
EVALS = "jailbreakbench,strongreject"      # add harmbench once you've accepted its HF terms
N_PROMPTS = 15                              # 0 = full benchmark. Start small to work out kinks.
SCORER = "rule_based"                       # free. Use "llm_judge" for higher fidelity (costs a little API credit).

In [ ]:
# 2. Secrets (set these in Colab's Secrets panel, not typed here)
from google.colab import userdata
import os

os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass

In [ ]:
# 3. Clone your repo (must already exist on GitHub -- see README for one-time setup) and install deps
import re
repo_name = GITHUB_REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
auth_url = re.sub(r"https://", f"https://{os.environ['GITHUB_TOKEN']}@", GITHUB_REPO_URL)

!git clone {auth_url}
%cd {repo_name}
!pip install -q -r requirements.txt

In [ ]:
# 4. Run the eval(s)
!python src/run_eval.py --model "{MODEL_ID}" --eval "{EVALS}" --n {N_PROMPTS} --scorer "{SCORER}"

In [ ]:
# 5. Build the readable markdown report from the raw JSON
!python src/report.py

In [ ]:
# 6. Commit + push results back to GitHub -- full transparency, prompts/responses/scores included
!git config user.email "colab@example.com"
!git config user.name "colab-eval-runner"
!git add results/
!git commit -m "eval run: {MODEL_ID} on {EVALS} ({SCORER})"
!git push